# NB_SLV_POI_ENTITY_NON_ENRICH

RECOMPUTE `lh_vv_silver.dbo.slv_poi_entity_non_enrich`: dòng google từ `slv_3p_poi` cộng dòng partner có bản i18n `vi`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# BEGIN_PARTNER_JSON
import json

JSON_OK, JSON_NO_VALUE, JSON_NULL, JSON_INVALID = "OK", "NO_VALUE", "NULL_JSON", "INVALID_JSON"
MAX_JSON_DEPTH = 64


def _first_text(value):
    """Chuỗi đầu tiên khác rỗng (đã trim). Mảng -> phần tử chuỗi đầu tiên khác rỗng; kiểu khác -> None."""
    for item in value if isinstance(value, list) else [value]:
        if isinstance(item, str) and item.strip():
            return item.strip()
    return None


def first_child_value(text, key):
    """Giá trị đầu tiên của khoá con `key` ở mọi độ sâu, theo thứ tự trong tài liệu (duyệt sâu trước).

    Duyệt bằng stack. Sâu hơn MAX_JSON_DEPTH, hoặc json.loads ném RecursionError, trả JSON_INVALID.
    Input : text — chuỗi JSON hoặc None ; key — tên khoá con
    Output: (giá trị | None, OK | NO_VALUE | NULL_JSON | INVALID_JSON)
    Ví dụ : first_child_value('{"facilities": {"classification": {"main_category": ["Hotel"]}}}', "main_category")
            -> ("Hotel", "OK")
    """
    if text is None:
        return None, JSON_NULL
    try:
        doc = json.loads(text)
    except (TypeError, ValueError, RecursionError):
        return None, JSON_INVALID
    if doc is None:
        return None, JSON_NULL

    stack = [("visit", doc, 1)]
    while stack:
        item = stack.pop()
        if item[0] == "visit":
            _, node, depth = item
            if depth > MAX_JSON_DEPTH:
                return None, JSON_INVALID
            if isinstance(node, dict):
                for name, value in reversed(list(node.items())):
                    stack.append(("pair", name, value, depth))
            elif isinstance(node, list):
                for child in reversed(node):
                    stack.append(("visit", child, depth + 1))
            continue
        _, name, value, depth = item
        if name == key:
            found = _first_text(value)
            if found is not None:
                return found, JSON_OK
        stack.append(("visit", value, depth + 1))
    return None, JSON_NO_VALUE
# END_PARTNER_JSON


In [ ]:
GOOGLE = "google"
PARTNER = "partner"
OTHER = "other"
OTHER_GROUP = "Other"
POI_COLS = ("poi_id", "source_name", "source_id", "language_code", "poi_name", "poi_name_normalized", "types_json",
            "operating_status", "primary_type", "_crawled_at", "_event_id", "_first_seen_at", "_last_seen_at")
ADDRESS_COLS = ("house_number", "street", "ward", "city", "full_address", "lat", "lng")
CATEGORY_COLS = ("business_sector", "business_category", "business_category_vi", "category_group")
OUT_COLS = POI_COLS + ADDRESS_COLS + CATEGORY_COLS
PARTNER_ADDRESS = {"house_number": "address_line1", "street": "street", "ward": "ward", "city": "city",
                   "full_address": "full_address"}
main_category_of = F.udf(lambda text: first_child_value(text, "main_category"), "struct<value:string,status:string>")


def _live(df: DataFrame) -> DataFrame:
    # Partner L1 CDC: deleted = true là dòng đã xoá ở nguồn (A2).
    return df.where(~F.coalesce(F.col("deleted"), F.lit(False)))


def _google(ctx: NodeContext, poi: DataFrame, d: DataFrame) -> DataFrame:
    # D7: primary_type -> phần tử đầu tiên của types_json khớp (theo vị trí) -> other / other / NULL / Other.
    g = (d.where(F.lower(F.trim("source_name")) == GOOGLE)
         .select(F.lower(F.trim("place_type")).alias("_k"), F.struct(*CATEGORY_COLS).alias("_cat")))
    dup = g.groupBy("_k").count().where("count > 1").count()
    if dup:
        raise QualityGateError(f"ref_dictionary_place_type: {dup} place_type google trùng")
    g = F.broadcast(g)
    by_primary = (poi.select("poi_id", F.lower(F.trim("primary_type")).alias("_k"))
                  .join(g, "_k").select("poi_id", F.col("_cat").alias("_p")))
    by_types = (poi.select("poi_id", F.posexplode(F.from_json("types_json", "array<string>")).alias("_pos", "_t"))
                .withColumn("_k", F.lower(F.trim("_t"))).join(g, "_k")
                .groupBy("poi_id").agg(F.min_by("_cat", "_pos").alias("_m")))
    addr = ctx.input("address").select("poi_id", *ADDRESS_COLS, F.lit(1).alias("_a"))
    out = poi.join(by_primary, "poi_id", "left").join(by_types, "poi_id", "left").join(addr, "poi_id", "left")
    agg = out.agg(F.count(F.lit(1)).alias("rows"), F.count("_p").alias("by_primary"),
                  F.count(F.when(F.col("_p").isNull(), F.col("_m"))).alias("by_types"),
                  F.count("_a").alias("with_address")).first()
    ctx.metrics.update(google_rows=agg["rows"], by_primary=agg["by_primary"], by_types=agg["by_types"],
                       unmatched=agg["rows"] - agg["by_primary"] - agg["by_types"],
                       google_with_address=agg["with_address"], dict_google_rows=g.count())

    def pick(c, default):
        return (F.when(F.col("_p").isNotNull(), F.col(f"_p.{c}"))
                .when(F.col("_m").isNotNull(), F.col(f"_m.{c}"))
                .otherwise(default).alias(c))

    return out.select(*POI_COLS, *ADDRESS_COLS, pick("business_sector", F.lit(OTHER)),
                      pick("business_category", F.lit(OTHER)),
                      pick("business_category_vi", F.lit(None).cast("string")),
                      pick("category_group", F.lit(OTHER_GROUP)))


def _partner(ctx: NodeContext, d: DataFrame, schema) -> DataFrame:
    svc = _live(ctx.input("svc"))
    vi = _live(ctx.input("i18n")).where(F.col("locale") == "vi").select(F.col("business_service_id").alias("id"))
    base = (svc.join(vi, "id").withColumn("_mc", main_category_of(F.col("amentities")))
            .withColumn("primary_type", F.col("_mc.value")))
    # business_category: source_name = partner, place_type = primary_type, khớp đúng chữ (A6).
    dp = (d.where(F.lower(F.trim("source_name")) == PARTNER)
          .select(F.col("place_type").alias("primary_type"),
                  F.struct("business_category", "business_category_vi").alias("_pc")))
    if dp.groupBy("primary_type").count().where("count > 1").count():
        raise QualityGateError("ref_dictionary_place_type: place_type partner trùng")
    # D3 bước 1: place_type = primary_type, không lọc source_name.
    s1 = d.groupBy(F.col("place_type").alias("primary_type")).agg(F.collect_set("business_category").alias("_bcs"))
    # D3 bước 2 + D9: dòng google cùng business_category; Other là phần còn lại, không tính là một nhóm.
    s2 = (d.where(F.lower(F.trim("source_name")) == GOOGLE)
          .groupBy(F.col("business_category").alias("_bc"))
          .agg(F.collect_set(F.when(F.col("category_group") != OTHER_GROUP, F.col("category_group"))).alias("_gs")))
    out = (base.join(F.broadcast(dp), "primary_type", "left").join(F.broadcast(s1), "primary_type", "left")
           .withColumn("_bc", F.when(F.size("_bcs") == 1, F.element_at("_bcs", 1)))
           .join(F.broadcast(s2), "_bc", "left")).persist()
    ctx.cached.append(out)
    live = svc.count()
    agg = out.agg(F.count(F.lit(1)).alias("rows"), F.count("primary_type").alias("primary"),
                  F.count("_pc").alias("primary_matched"),
                  F.count(F.when(F.size("_bcs") > 1, 1)).alias("amb_step1"),
                  F.count(F.when(F.size("_gs") > 1, 1)).alias("amb_step2"),
                  F.count(F.when(F.size("_gs") == 1, 1)).alias("named_group")).first()
    status = {r["s"]: r["n"] for r in out.groupBy(F.col("_mc.status").alias("s")).count()
              .withColumnRenamed("count", "n").collect()}
    ctx.metrics.update(partner_live=live, partner_rows=agg["rows"], partner_without_vi=live - agg["rows"],
                       partner_primary=agg["primary"], partner_primary_matched=agg["primary_matched"],
                       partner_named_group=agg["named_group"], partner_main_category=status)
    if agg["amb_step1"] or agg["amb_step2"]:
        raise QualityGateError(f"ref_dictionary_place_type: D3 không xác định category_group "
                               f"(bước 1: {agg['amb_step1']}, bước 2: {agg['amb_step2']} dòng partner)")
    if live and not agg["rows"]:
        raise QualityGateError("slv_pn_business_services: không dòng nào có bản i18n vi")
    no_type = F.col("primary_type").isNull()
    cols = {
        "poi_id": uuid_md5(F.col("id"), F.lit(PARTNER)),  # D2
        "source_name": F.lit(PARTNER),
        "source_id": F.col("id"),
        "language_code": F.lit("vi"),
        "poi_name": F.col("name"),
        "poi_name_normalized": F.lower(F.col("name")),
        "operating_status": F.col("operation_status"),  # D4
        "primary_type": F.col("primary_type"),
        "business_sector": F.col("business_sector"),
        "business_category": F.when(no_type, F.col("business_sector")).otherwise(F.col("_pc.business_category")),
        "business_category_vi": F.when(no_type, F.col("business_sector")).otherwise(F.col("_pc.business_category_vi")),
        "category_group": F.when(F.size("_gs") == 1, F.element_at("_gs", 1)).otherwise(F.lit(OTHER_GROUP)),
        **{c: F.get_json_object("address", f"$.{k}") for c, k in PARTNER_ADDRESS.items()},
    }
    return out.select(*[cols.get(c, F.lit(None)).cast(schema[c].dataType).alias(c) for c in OUT_COLS])


def build(ctx: NodeContext) -> DataFrame:
    src = ctx.input("poi")
    missing = [c for c in POI_COLS if c not in src.columns]
    if missing:
        raise ConfigError(f"slv_poi_entity_non_enrich: slv_3p_poi thiếu cột {missing}")
    d = ctx.input("dict")
    if "category_group" not in d.columns:
        raise ConfigError("ref_dictionary_place_type chưa có category_group: chạy cell [THÊM 19/10] của NB_CREATE_DDL")
    d = d.where(F.col("is_active") == F.lit(True))
    no_group = d.where(F.col("category_group").isNull()).count()
    if no_group:
        raise QualityGateError(f"ref_dictionary_place_type: {no_group} dòng active thiếu category_group")
    total = src.count()
    google = _google(ctx, src.where(F.lower(F.trim("source_name")) == GOOGLE).select(*POI_COLS), d)
    ctx.metrics.update(non_google_rows=total - ctx.metrics["google_rows"])
    if total and not ctx.metrics["google_rows"]:
        raise QualityGateError("slv_poi_entity_non_enrich: slv_3p_poi không có dòng google")
    return google.unionByName(_partner(ctx, d, google.schema))


SPEC = NodeSpec(
    trg_tbl="slv_poi_entity_non_enrich",
    inputs={"poi": "slv_3p_poi", "address": "slv_3p_poi_address", "dict": "ref_dictionary_place_type",
            "svc": "slv_pn_business_services", "i18n": "slv_pn_business_service_i18ns"},
    pk=("poi_id",),
    build=build,
    required_inputs=("poi", "address", "dict", "svc", "i18n"),
    unique_checks=(("partner_source_id_unique", ("source_id",), "source_name = 'partner'"),),
    warn_checks=(("types_json_unparsed",
                  "types_json IS NOT NULL AND from_json(types_json, 'array<string>') IS NULL"),
                 ("partner_primary_type_unmatched",
                  "source_name = 'partner' AND primary_type IS NOT NULL AND business_category IS NULL")),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
